In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
"""
=============================================================================
MAPA-CLIP-FDB: Metadata-Aligned Prompt Adaptation for CLIP
             + Fuzzy Dual-Branch Image Encoder (Frozen DenseNet-121)

NOVEL ALIGNMENT: Distributional Harmony via Optimal Transport + HSIC
─────────────────────────────────────────────────────────────────────────────
  Instead of contrastive loss, we use:
  1. Optimal Transport (Sinkhorn) to measure distributional alignment
  2. HSIC to maximize statistical dependence within classes
  3. Class-conditional moment matching for semantic consistency

SEEDING STRATEGY — Option B: Random support set per run, fixed full test set
─────────────────────────────────────────────────────────────────────────────
  GLOBAL_SEED (fixed=42):
    • 80/20 train/test stratified split  → same pool every run
    • Full test set is ALWAYS the same   → fair comparison across runs

  RUN_SEED (random, time-based):
    • Few-shot support set selection     → different 5 images/class each run
    • Model weight initialisation        → different starting point each run
=============================================================================
"""

import os
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)
import clip
from PIL import Image
import torchvision.models as tv_models
import torchvision.transforms as T
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================
DATA_ROOT  = "/kaggle/input/pad-ufes-20/PAD-UFES-20"
CSV_PATH   = os.path.join(DATA_ROOT, "metadata.csv")
IMG_DIR    = os.path.join(DATA_ROOT, "Dataset")
OUTPUT_DIR = "/kaggle/working"

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_ARCH = "ViT-B/16"

N_SHOTS = 5

# ── Seeding Strategy ─────────────────────────────────────────────────────────
GLOBAL_SEED = 42
RUN_SEED    = int(time.time()) % 100_000

print(f"★ GLOBAL_SEED (fixed)  : {GLOBAL_SEED}  ← train/test split always identical")
print(f"★ RUN_SEED    (random) : {RUN_SEED}  ← support set & model init change each run")

N_CTX      = 16
LATENT_DIM = 256
CLIP_DIM   = 512
DENSE_DIM  = 512

FUZZY_ALPHA = 1.0
FUZZY_SIGMA = 0.5

EPOCHS        = 200
LR_PROMPT     = 2e-3
LR_META       = 1e-4
LR_DENSE_HEAD = 1e-4

# ★ NEW: Optimal Transport + HSIC parameters
LAMBDA_OT     = 0.3   # Optimal Transport alignment weight
LAMBDA_HSIC   = 0.2   # HSIC dependence maximization weight
LAMBDA_MOMENT = 0.15  # Moment matching weight
LAMBDA_RECON  = 0.2   # Reconstruction weight
LAMBDA_TRIP   = 0.1   # Triplet weight

OT_REG        = 0.05  # Sinkhorn regularization (entropy)
OT_ITERS      = 50    # Sinkhorn iterations

MARGIN        = 1.0
TEST_BATCH    = 32

CLASS_FULLNAME_MAP = {
    "ACK": "actinic keratosis skin lesion",
    "BCC": "basal cell carcinoma skin cancer",
    "MEL": "melanoma skin cancer",
    "NEV": "nevus benign mole",
    "SCC": "squamous cell carcinoma skin cancer",
    "SEK": "seborrheic keratosis skin lesion",
}

print(f"\nDevice : {DEVICE}")
print(f"CLIP   : {CLIP_ARCH}  (Branch B — ViT, FROZEN)")
print(f"CNN    : DenseNet-121 (Branch A — backbone FROZEN, head trainable)")
print(f"\n★ NOVEL ALIGNMENT: Optimal Transport + HSIC (NO contrastive loss)")

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_csv(CSV_PATH)

img_col   = "img_id"      if "img_id"      in df.columns else df.columns[0]
label_col = "diagnostic"  if "diagnostic"  in df.columns else df.columns[-1]
df        = df.reset_index(drop=True)

classes   = sorted(df[label_col].unique().tolist())
N_CLASSES = len(classes)
c2i       = {c: i for i, c in enumerate(classes)}
i2c       = {i: c for c, i in c2i.items()}

print(f"\n{'='*60}\nDATASET OVERVIEW\n{'='*60}")
print(f"Total samples : {len(df)}")
print(f"Classes ({N_CLASSES}) : {classes}")
for c in classes:
    n  = (df[label_col] == c).sum()
    fn = CLASS_FULLNAME_MAP.get(c, c)
    print(f"  [{c}] {fn:40s}: {n:4d} samples")

clip_classnames = [CLASS_FULLNAME_MAP.get(c, c.lower()) for c in classes]
print(f"\nCLIP class names: {clip_classnames}")

# =============================================================================
# 2. METADATA PROCESSING
# =============================================================================
BINARY_COLS = [c for c in [
    "smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history",
    "has_piped_water", "has_sewage_system", "itch", "grew", "hurt",
    "changed", "bleed", "elevation", "biopsed",
] if c in df.columns]

CONT_COLS = [c for c in [
    "age", "diameter_1", "diameter_2", "fitspatrick",
] if c in df.columns]

CAT_COLS = [c for c in [
    "gender", "region", "background_father", "background_mother",
] if c in df.columns]

BOOL_MAP = {
    True: 1.0, False: 0.0, "True": 1.0, "False": 0.0,
    "true": 1.0, "false": 0.0,
}

print(f"\n{'='*60}\nMETADATA PROCESSING\n{'='*60}")
print(f"  Binary features      ({len(BINARY_COLS)}): {BINARY_COLS}")
print(f"  Continuous features  ({len(CONT_COLS)}): {CONT_COLS}")
print(f"  Categorical features ({len(CAT_COLS)}): {CAT_COLS}")

if CONT_COLS:
    _cont_raw  = df[CONT_COLS].replace("UNK", np.nan).apply(pd.to_numeric, errors="coerce")
    _col_means = _cont_raw.mean()
    _scaler    = StandardScaler()
    _scaler.fit(_cont_raw.fillna(_col_means))
else:
    _scaler, _col_means = None, None

_cat_ref_cols = (
    pd.get_dummies(df[CAT_COLS].fillna("UNK").astype(str), dtype=float).columns.tolist()
    if CAT_COLS else []
)

def build_meta_tensor(dataframe):
    parts = []
    idx   = dataframe.index
    if BINARY_COLS:
        b = (dataframe[BINARY_COLS].replace("UNK", np.nan)
             .replace(BOOL_MAP).apply(pd.to_numeric, errors="coerce").fillna(-1.0))
        parts.append(b)
    if CONT_COLS and _scaler is not None:
        c  = (dataframe[CONT_COLS].replace("UNK", np.nan)
              .apply(pd.to_numeric, errors="coerce").fillna(_col_means))
        cs = _scaler.transform(c)
        parts.append(pd.DataFrame(cs, index=idx, columns=CONT_COLS))
    if CAT_COLS:
        cat = pd.get_dummies(dataframe[CAT_COLS].fillna("UNK").astype(str), dtype=float)
        cat = cat.reindex(columns=_cat_ref_cols, fill_value=0.0)
        parts.append(cat)
    merged = pd.concat(parts, axis=1).reset_index(drop=True)
    return torch.tensor(merged.values, dtype=torch.float32)

META_TENSOR  = build_meta_tensor(df)
LABEL_TENSOR = torch.tensor([c2i[c] for c in df[label_col]], dtype=torch.long)
META_DIM     = META_TENSOR.shape[1]
print(f"\n  Final metadata dimension: {META_DIM}")

# =============================================================================
# 3. TRAIN / TEST SPLIT — FIXED with GLOBAL_SEED
# =============================================================================
ALL_IDX = np.arange(len(df))
TRAIN_IDX, TEST_IDX = train_test_split(
    ALL_IDX, test_size=0.2, stratify=df[label_col].values,
    random_state=GLOBAL_SEED,
)

print(f"\n{'='*60}\nDATA SPLITS\n{'='*60}")
print(f"  Train pool (80%) : {len(TRAIN_IDX):5d} samples  [GLOBAL_SEED={GLOBAL_SEED}, same every run]")
print(f"  Test  set  (20%) : {len(TEST_IDX):5d} samples  [GLOBAL_SEED={GLOBAL_SEED}, same every run]")
print(f"\n  Test class distribution (fixed across all runs):")
for c in classes:
    n = (df.iloc[TEST_IDX][label_col].values == c).sum()
    print(f"    [{c}]: {n}")

# =============================================================================
# 4. FEW-SHOT SAMPLING — RANDOM with RUN_SEED
# =============================================================================
rng_run = np.random.RandomState(RUN_SEED)
FS_IDX  = []
for cls in classes:
    mask   = df.iloc[TRAIN_IDX][label_col].values == cls
    pool   = TRAIN_IDX[mask]
    chosen = rng_run.choice(pool, size=min(N_SHOTS, len(pool)), replace=False)
    FS_IDX.extend(chosen.tolist())
FS_IDX = np.array(FS_IDX)

print(f"\n{'='*60}\nFEW-SHOT SUPPORT SET\n{'='*60}")
print(f"  {N_SHOTS}/class × {N_CLASSES} classes = {len(FS_IDX)} total")
print(f"  [RUN_SEED={RUN_SEED} — DIFFERENT images selected this run]")
for c in classes:
    idxs = FS_IDX[(df.iloc[FS_IDX][label_col].values == c)]
    imgs = df.iloc[idxs][img_col].values.tolist()
    print(f"    [{c}]: {imgs}")

# =============================================================================
# 5. DATASET
# =============================================================================
class PADDataset(Dataset):
    def __init__(self, indices, img_dir, clip_transform=None, dense_transform=None):
        self.indices         = indices
        self.meta            = META_TENSOR[indices]
        self.labels          = LABEL_TENSOR[indices]
        self.img_dir         = img_dir
        self.clip_transform  = clip_transform
        self.dense_transform = dense_transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        orig_idx = self.indices[i]
        img_id   = df.iloc[orig_idx][img_col]
        path     = os.path.join(self.img_dir, img_id)
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            found = False
            for root, _, files in os.walk(self.img_dir):
                if img_id in files:
                    img = Image.open(os.path.join(root, img_id)).convert("RGB")
                    found = True
                    break
            if not found:
                img = Image.new("RGB", (224, 224), (128, 128, 128))
        clip_img  = self.clip_transform(img)  if self.clip_transform  else img
        dense_img = self.dense_transform(img) if self.dense_transform else img
        return clip_img, dense_img, self.meta[i], self.labels[i]

# =============================================================================
# 6. LOAD CLIP (Branch B — fully frozen)
# =============================================================================
print(f"\n{'='*60}\nLOADING CLIP ({CLIP_ARCH}) — Branch B: ViT (FROZEN)\n{'='*60}")
clip_model, clip_preprocess = clip.load(CLIP_ARCH, device=DEVICE)
clip_model = clip_model.float()
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)
print("  CLIP ViT-B/16 loaded, frozen, cast to float32.")

# =============================================================================
# 7. DENSENET-121 PREPROCESSING (Branch A — backbone frozen)
# =============================================================================
dense_preprocess = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# =============================================================================
# 8. MODEL COMPONENTS
# =============================================================================

class PromptLearner(nn.Module):
    def __init__(self, classnames, clip_model, n_ctx=16):
        super().__init__()
        n_cls   = len(classnames)
        ctx_dim = clip_model.ln_final.weight.shape[0]
        ctx     = torch.empty(n_ctx, ctx_dim)
        nn.init.normal_(ctx, std=0.02)
        self.ctx = nn.Parameter(ctx)
        ref_texts = [" ".join(["X"] * n_ctx) + " " + cn + "." for cn in classnames]
        tokenized = clip.tokenize(ref_texts).to(DEVICE)
        with torch.no_grad():
            emb = clip_model.token_embedding(tokenized).float()
        self.register_buffer("prefix", emb[:, :1,        :])
        self.register_buffer("suffix", emb[:, 1 + n_ctx:, :])
        self.tokenized      = tokenized
        self.n_cls, self.n_ctx = n_cls, n_ctx
        print(f"\n  PromptLearner  n_ctx={n_ctx}  classes={classnames}")

    def forward(self):
        ctx = self.ctx.unsqueeze(0).expand(self.n_cls, -1, -1)
        return torch.cat([self.prefix, ctx, self.suffix], dim=1)


class CLIPTextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.pos_emb     = clip_model.positional_embedding
        self.ln_final    = clip_model.ln_final
        self.text_proj   = clip_model.text_projection

    def forward(self, prompts, tokenized):
        x = prompts + self.pos_emb
        x = self.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = self.ln_final(x)
        return (x[torch.arange(x.shape[0]), tokenized.argmax(dim=-1)] @ self.text_proj).float()


class DenseNetBranch(nn.Module):
    """DenseNet-121 with FULLY FROZEN backbone."""
    def __init__(self, out_dim=DENSE_DIM):
        super().__init__()
        base        = tv_models.densenet121(weights=tv_models.DenseNet121_Weights.IMAGENET1K_V1)
        in_features = base.classifier.in_features

        for param in base.features.parameters():
            param.requires_grad_(False)

        base.classifier = nn.Identity()
        self.backbone   = base

        self.proj_head = nn.Sequential(
            nn.Linear(in_features, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, out_dim),
        )

    def forward(self, x):
        with torch.no_grad():
            feat = self.backbone.features(x)
            feat = F.relu(feat, inplace=True)
            feat = F.adaptive_avg_pool2d(feat, (1, 1))
            feat = feat.flatten(1)
        return self.proj_head(feat)


class FuzzyDualBranchFusion(nn.Module):
    """Gaussian fuzzy confidence-based adaptive fusion."""
    def __init__(self, feat_dim=CLIP_DIM, alpha=FUZZY_ALPHA, sigma=FUZZY_SIGMA):
        super().__init__()
        self.feat_dim = feat_dim
        self.alpha_c  = nn.Parameter(torch.tensor(alpha))
        self.sigma_c  = nn.Parameter(torch.tensor(sigma))
        self.alpha_t  = nn.Parameter(torch.tensor(alpha))
        self.sigma_t  = nn.Parameter(torch.tensor(sigma))

    def _gaussian_membership(self, x, alpha, sigma):
        sigma_sq = sigma.abs().clamp(min=1e-6) ** 2
        return torch.exp(-((x - alpha) ** 2) / sigma_sq)

    def forward(self, F_c, F_t):
        mu_c  = F_c.abs().mean(dim=-1, keepdim=True)
        mu_t  = F_t.abs().mean(dim=-1, keepdim=True)
        hc    = self._gaussian_membership(mu_c, self.alpha_c, self.sigma_c)
        ht    = self._gaussian_membership(mu_t, self.alpha_t, self.sigma_t)
        denom = hc + ht + 1e-8
        w_c, w_t = hc / denom, ht / denom
        fused = F.normalize(w_c * F_c + w_t * F_t, dim=-1)
        return fused, w_c, w_t


class MetadataEncoder(nn.Module):
    def __init__(self, in_dim, latent_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, in_dim),
        )

    def forward(self, x, recon=False):
        z = self.encoder(x)
        return (z, self.decoder(z)) if recon else z


class MetadataProjector(nn.Module):
    def __init__(self, in_dim=256, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 384), nn.LayerNorm(384), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(384, out_dim),
        )

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)


# =============================================================================
# 9. NOVEL ALIGNMENT FUNCTIONS
# =============================================================================

def sinkhorn_distance(X, Y, reg=OT_REG, num_iters=OT_ITERS):
    """
    Compute Optimal Transport distance using Sinkhorn algorithm.
    
    Args:
        X: [B, D] source features (vision)
        Y: [B, D] target features (text)
        reg: regularization parameter (entropy)
        num_iters: number of Sinkhorn iterations
    
    Returns:
        Wasserstein distance (scalar)
    """
    B, D = X.shape
    
    # Compute cost matrix (Euclidean distance squared)
    C = torch.cdist(X, Y, p=2) ** 2
    
    # Initialize uniform distributions
    a = torch.ones(B, device=X.device) / B
    b = torch.ones(B, device=X.device) / B
    
    # Sinkhorn iterations
    K = torch.exp(-C / reg)
    u = torch.ones(B, device=X.device) / B
    
    for _ in range(num_iters):
        v = b / (K.T @ u + 1e-8)
        u = a / (K @ v + 1e-8)
    
    # Compute transport plan and distance
    P = u.unsqueeze(1) * K * v.unsqueeze(0)
    distance = torch.sum(P * C)
    
    return distance


def hsic_loss(X, Y, labels):
    """
    Hilbert-Schmidt Independence Criterion for measuring dependence.
    Maximizes dependence within same class, minimizes across classes.
    
    Args:
        X: [B, D] vision features
        Y: [B, D] text features  
        labels: [B] class labels
    
    Returns:
        HSIC-based alignment loss
    """
    B = X.shape[0]
    
    # RBF kernel with median heuristic
    def rbf_kernel(A, sigma=None):
        pairwise_sq_dists = torch.cdist(A, A, p=2) ** 2
        if sigma is None:
            sigma = torch.median(pairwise_sq_dists).detach()
            sigma = sigma.clamp(min=1e-4)
        return torch.exp(-pairwise_sq_dists / (2 * sigma))
    
    # Compute kernels
    K_X = rbf_kernel(X)
    K_Y = rbf_kernel(Y)
    
    # Center kernels
    H = torch.eye(B, device=X.device) - torch.ones(B, B, device=X.device) / B
    K_X_centered = H @ K_X @ H
    K_Y_centered = H @ K_Y @ H
    
    # HSIC for same class pairs (maximize)
    same_class_mask = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    hsic_within = torch.sum(K_X_centered * K_Y_centered * same_class_mask) / (same_class_mask.sum() + 1e-8)
    
    # HSIC for different class pairs (minimize)
    diff_class_mask = 1.0 - same_class_mask
    hsic_between = torch.sum(K_X_centered * K_Y_centered * diff_class_mask) / (diff_class_mask.sum() + 1e-8)
    
    # We want to maximize within-class dependence and minimize between-class
    return -hsic_within + hsic_between


def class_conditional_moment_matching(X, Y, labels, n_classes):
    """
    Match statistical moments (mean, variance) of class-conditional distributions.
    
    Args:
        X: [B, D] vision features
        Y: [B, D] text features
        labels: [B] class labels
        n_classes: number of classes
    
    Returns:
        Moment matching loss
    """
    moment_loss = 0.0
    n_matched = 0
    
    for c in range(n_classes):
        mask = (labels == c)
        if mask.sum() < 2:  # Need at least 2 samples
            continue
            
        X_c = X[mask]
        Y_c = Y[mask]
        
        # Mean matching
        mean_loss = F.mse_loss(X_c.mean(dim=0), Y_c.mean(dim=0))
        
        # Variance matching
        var_X = X_c.var(dim=0, unbiased=False)
        var_Y = Y_c.var(dim=0, unbiased=False)
        var_loss = F.mse_loss(var_X, var_Y)
        
        moment_loss += (mean_loss + var_loss)
        n_matched += 1
    
    return moment_loss / max(n_matched, 1)


def distributional_harmony_loss(I, M, T, labels, n_classes):
    """
    Combined Optimal Transport + HSIC + Moment Matching alignment.
    
    Args:
        I: [B, D] fused image features
        M: [B, D] metadata features
        T: [C, D] text prototypes (C classes)
        labels: [B] ground truth labels
        n_classes: number of classes
    
    Returns:
        tuple: (ot_loss, hsic_loss_val, moment_loss)
    """
    B, D = I.shape
    
    # 1. Optimal Transport between image and corresponding text prototypes
    T_batch = T[labels]  # [B, D] - select text prototype for each sample's class
    ot_loss = sinkhorn_distance(I, T_batch, reg=OT_REG, num_iters=OT_ITERS)
    
    # 2. HSIC-based dependence (vision-text)
    hsic_loss_val = hsic_loss(I, T_batch, labels)
    
    # 3. Class-conditional moment matching (image-metadata consistency)
    moment_loss = class_conditional_moment_matching(I, M, labels, n_classes)
    
    return ot_loss, hsic_loss_val, moment_loss


# =============================================================================
# 10. MAPA-CLIP-FDB MODEL (with Novel Alignment)
# =============================================================================
class MAPACLIPFDBModel(nn.Module):
    def __init__(self, classnames, clip_model, meta_dim,
                 n_ctx=16, latent_dim=256, clip_dim=512):
        super().__init__()
        self.clip_model      = clip_model
        self.n_cls           = len(classnames)
        self.prompt_learner  = PromptLearner(classnames, clip_model, n_ctx)
        self.text_encoder    = CLIPTextEncoder(clip_model)
        self.densenet_branch = DenseNetBranch(out_dim=clip_dim)
        self.fuzzy_fusion    = FuzzyDualBranchFusion(feat_dim=clip_dim)
        self.meta_encoder    = MetadataEncoder(meta_dim, latent_dim)
        self.meta_projector  = MetadataProjector(latent_dim, clip_dim)
        self.alpha           = nn.Parameter(torch.tensor(0.5))
        self.log_scale       = clip_model.logit_scale

    @torch.no_grad()
    def _encode_vit(self, clip_imgs):
        return self.clip_model.encode_image(clip_imgs).float()

    def _encode_images(self, clip_imgs, dense_imgs):
        F_t = self._encode_vit(clip_imgs)
        F_c = self.densenet_branch(dense_imgs)
        I, w_c, w_t = self.fuzzy_fusion(F_c, F_t)
        return I, F_c, F_t, w_c, w_t

    def _encode_text(self):
        prompts   = self.prompt_learner()
        tokenized = self.prompt_learner.tokenized
        return F.normalize(self.text_encoder(prompts, tokenized), dim=-1)

    def _encode_meta(self, meta, recon=False):
        if recon:
            z, xr = self.meta_encoder(meta, recon=True)
            return self.meta_projector(z), z, xr
        z = self.meta_encoder(meta)
        return self.meta_projector(z), z

    def forward(self, clip_imgs, dense_imgs, meta, mode="eval"):
        I, F_c, F_t, w_c, w_t = self._encode_images(clip_imgs, dense_imgs)
        T = self._encode_text()
        if mode == "train":
            M, z, xr = self._encode_meta(meta, recon=True)
        else:
            M, z     = self._encode_meta(meta, recon=False)
            xr       = None
        tau    = self.log_scale.exp()
        L_it   = tau * (I @ T.T)
        L_im   = tau * (M @ T.T)
        alpha  = torch.sigmoid(self.alpha)
        L_fuse = alpha * L_it + (1.0 - alpha) * L_im
        if mode == "train":
            return L_fuse, L_it, L_im, I, M, T, z, xr, w_c, w_t
        return L_fuse, L_it, L_im


# =============================================================================
# 11. ADDITIONAL LOSS FUNCTIONS
# =============================================================================
def triplet_loss_fn(z, labels, margin=MARGIN):
    B, dev = z.size(0), z.device
    arange = torch.arange(B, device=dev)
    a_list, p_list, n_list = [], [], []
    for i in range(B):
        pi = torch.where((labels == labels[i]) & (arange != i))[0]
        ni = torch.where(labels != labels[i])[0]
        if len(pi) == 0 or len(ni) == 0:
            continue
        p = pi[torch.randint(len(pi), (1,), device=dev)]
        n = ni[torch.randint(len(ni), (1,), device=dev)]
        a_list.append(z[i:i+1]); p_list.append(z[p]); n_list.append(z[n])
    if not a_list:
        return z.sum() * 0.0
    a, p, n = torch.cat(a_list), torch.cat(p_list), torch.cat(n_list)
    return F.relu(F.pairwise_distance(a, p) - F.pairwise_distance(a, n) + margin).mean()


# =============================================================================
# 12. DATA LOADERS
# =============================================================================
fs_loader = DataLoader(
    PADDataset(FS_IDX,   IMG_DIR, clip_preprocess, dense_preprocess),
    batch_size=len(FS_IDX), shuffle=True,
)
test_loader = DataLoader(
    PADDataset(TEST_IDX, IMG_DIR, clip_preprocess, dense_preprocess),
    batch_size=TEST_BATCH, shuffle=False,
)

# =============================================================================
# 13. MODEL INIT & OPTIMISERS
# =============================================================================
print(f"\n{'='*60}\nMAPA-CLIP-FDB MODEL INITIALISATION\n{'='*60}")

torch.manual_seed(RUN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RUN_SEED)

model = MAPACLIPFDBModel(
    classnames=clip_classnames, clip_model=clip_model,
    meta_dim=META_DIM, n_ctx=N_CTX, latent_dim=LATENT_DIM, clip_dim=CLIP_DIM,
).to(DEVICE)

prompt_params     = list(model.prompt_learner.parameters())
dense_head_params = (
    list(model.densenet_branch.proj_head.parameters())
    + list(model.fuzzy_fusion.parameters())
)
meta_params = (
    list(model.meta_encoder.parameters())
    + list(model.meta_projector.parameters())
    + [model.alpha]
)

n_prompt     = sum(p.numel() for p in prompt_params)
n_dense_head = sum(p.numel() for p in dense_head_params)
n_meta       = sum(p.numel() for p in meta_params)
n_dense_back = sum(p.numel() for p in model.densenet_branch.backbone.parameters())

print(f"\n  ── Trainable ──────────────────────────────────")
print(f"  Prompt context tokens      : {n_prompt:>12,}")
print(f"  DenseNet proj head + Fuzzy : {n_dense_head:>12,}")
print(f"  Metadata modules           : {n_meta:>12,}")
print(f"  Total trainable            : {n_prompt+n_dense_head+n_meta:>12,}")
print(f"\n  ── Frozen ─────────────────────────────────────")
print(f"  CLIP ViT-B/16              : {sum(p.numel() for p in clip_model.parameters()):>12,}")
print(f"  DenseNet-121 backbone      : {n_dense_back:>12,}  ★ FROZEN")

opt_prompt     = torch.optim.AdamW(prompt_params,     lr=LR_PROMPT,     weight_decay=1e-4)
opt_dense_head = torch.optim.AdamW(dense_head_params, lr=LR_DENSE_HEAD, weight_decay=1e-4)
opt_meta       = torch.optim.AdamW(meta_params,       lr=LR_META,       weight_decay=1e-4)

sch_prompt     = torch.optim.lr_scheduler.CosineAnnealingLR(opt_prompt,     T_max=EPOCHS, eta_min=1e-5)
sch_dense_head = torch.optim.lr_scheduler.CosineAnnealingLR(opt_dense_head, T_max=EPOCHS, eta_min=1e-6)
sch_meta       = torch.optim.lr_scheduler.CosineAnnealingLR(opt_meta,       T_max=EPOCHS, eta_min=1e-6)

CE  = nn.CrossEntropyLoss(label_smoothing=0.1)
MSE = nn.MSELoss()

# =============================================================================
# 14. ZERO-SHOT CLIP BASELINE
# =============================================================================
print(f"\n{'='*60}\nZERO-SHOT CLIP BASELINE\n{'='*60}")
zs_templates = [f"a clinical dermatology image of {cn}" for cn in clip_classnames]
with torch.no_grad():
    zs_txt_feat = F.normalize(
        clip_model.encode_text(clip.tokenize(zs_templates).to(DEVICE)).float(), dim=-1
    )

zs_preds, zs_labs = [], []
for clip_imgs, dense_imgs, _, labs in test_loader:
    with torch.no_grad():
        img_f = F.normalize(clip_model.encode_image(clip_imgs.to(DEVICE)).float(), dim=-1)
        sims  = (100.0 * img_f @ zs_txt_feat.T).argmax(dim=-1)
    zs_preds.extend(sims.cpu().numpy()); zs_labs.extend(labs.numpy())

zs_preds = np.array(zs_preds); zs_labs = np.array(zs_labs)
print(f"  Accuracy     : {accuracy_score(zs_labs, zs_preds):.4f}")
print(f"  Balanced Acc : {balanced_accuracy_score(zs_labs, zs_preds):.4f}")
print(f"  Macro F1     : {f1_score(zs_labs, zs_preds, average='macro'):.4f}")

# =============================================================================
# 15. TRAINING LOOP (with Novel Alignment)
# =============================================================================
print(f"\n{'='*60}")
print(f"TRAINING with NOVEL ALIGNMENT (Optimal Transport + HSIC)")
print(f"{'='*60}")
print(f"  LR_PROMPT={LR_PROMPT}  LR_DENSE_HEAD={LR_DENSE_HEAD}  LR_META={LR_META}")
print(f"  λ_OT={LAMBDA_OT}  λ_HSIC={LAMBDA_HSIC}  λ_moment={LAMBDA_MOMENT}")
print(f"  λ_recon={LAMBDA_RECON}  λ_trip={LAMBDA_TRIP}")

best_acc, best_ep = 0.0, 0
hist = {k: [] for k in ["loss","cls","ot","hsic","moment","recon","trip","acc","w_cnn_mean","w_vit_mean"]}

for ep in range(EPOCHS):
    model.train()
    model.densenet_branch.backbone.eval()

    ep_wc, ep_wt = [], []
    for clip_imgs, dense_imgs, meta, labs in fs_loader:
        clip_imgs  = clip_imgs.to(DEVICE)
        dense_imgs = dense_imgs.to(DEVICE)
        meta       = meta.to(DEVICE)
        labs       = labs.to(DEVICE)

        L_fuse, L_it, L_im, I, M, T, z, xr, w_c, w_t = model(
            clip_imgs, dense_imgs, meta, mode="train"
        )
        ep_wc.append(w_c.mean().item())
        ep_wt.append(w_t.mean().item())

        # Classification loss
        l_cls   = CE(L_fuse, labs) + 0.5 * CE(L_it, labs)
        
        # ★ NOVEL ALIGNMENT: Distributional Harmony
        ot_loss, hsic_loss_val, moment_loss = distributional_harmony_loss(
            I, M, T, labs, N_CLASSES
        )
        
        # Auxiliary losses
        l_recon = MSE(xr, meta)
        l_trip  = triplet_loss_fn(z, labs)
        
        # Total loss
        loss = (l_cls + 
                LAMBDA_OT * ot_loss + 
                LAMBDA_HSIC * hsic_loss_val + 
                LAMBDA_MOMENT * moment_loss +
                LAMBDA_RECON * l_recon + 
                LAMBDA_TRIP * l_trip)

        opt_prompt.zero_grad(); opt_dense_head.zero_grad(); opt_meta.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(prompt_params,     max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(dense_head_params, max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(meta_params,       max_norm=1.0)
        opt_prompt.step(); opt_dense_head.step(); opt_meta.step()

    sch_prompt.step(); sch_dense_head.step(); sch_meta.step()

    hist["loss"].append(loss.item())
    hist["cls"].append(l_cls.item())
    hist["ot"].append(ot_loss.item())
    hist["hsic"].append(hsic_loss_val.item())
    hist["moment"].append(moment_loss.item())
    hist["recon"].append(l_recon.item())
    hist["trip"].append(l_trip.item() if isinstance(l_trip, torch.Tensor) else 0.0)
    hist["w_cnn_mean"].append(np.mean(ep_wc))
    hist["w_vit_mean"].append(np.mean(ep_wt))

    model.eval()
    ep_preds, ep_labs = [], []
    with torch.no_grad():
        for clip_imgs, dense_imgs, meta, labs in test_loader:
            Lf, _, _ = model(clip_imgs.to(DEVICE), dense_imgs.to(DEVICE),
                             meta.to(DEVICE), mode="eval")
            ep_preds.extend(Lf.argmax(1).cpu().numpy())
            ep_labs.extend(labs.numpy())

    acc = accuracy_score(ep_labs, ep_preds)
    hist["acc"].append(acc)

    if acc > best_acc:
        best_acc = acc; best_ep = ep + 1
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_mapa_clip_fdb_run{RUN_SEED}.pth")

    if (ep + 1) % 25 == 0 or ep == 0:
        print(
            f"  Ep [{ep+1:3d}/{EPOCHS}] loss={loss.item():.4f} "
            f"(cls={l_cls.item():.3f} ot={ot_loss.item():.3f} "
            f"hsic={hsic_loss_val.item():.3f} mom={moment_loss.item():.3f} "
            f"rec={l_recon.item():.3f} trp={hist['trip'][-1]:.3f}) "
            f"w_cnn={np.mean(ep_wc):.3f} w_vit={np.mean(ep_wt):.3f} "
            f"| Acc={acc:.4f}  [Best={best_acc:.4f} @ep{best_ep}]"
        )

print(f"\nTraining complete.  Best accuracy: {best_acc:.4f}  (epoch {best_ep})")

# =============================================================================
# 16. FINAL EVALUATION
# =============================================================================
print(f"\n{'='*60}\nFINAL EVALUATION\n{'='*60}")

model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_mapa_clip_fdb_run{RUN_SEED}.pth"))
model.eval()

all_preds, all_labs, all_probs = [], [], []
all_wc, all_wt = [], []

with torch.no_grad():
    for clip_imgs, dense_imgs, meta, labs in test_loader:
        clip_imgs  = clip_imgs.to(DEVICE)
        dense_imgs = dense_imgs.to(DEVICE)
        meta_d     = meta.to(DEVICE)
        Lf, _, _ = model(clip_imgs, dense_imgs, meta_d, mode="eval")
        probs = F.softmax(Lf, dim=-1)
        all_preds.extend(Lf.argmax(1).cpu().numpy())
        all_labs.extend(labs.numpy())
        all_probs.extend(probs.cpu().numpy())
        I, F_c, F_t, w_c, w_t = model._encode_images(clip_imgs, dense_imgs)
        all_wc.extend(w_c.squeeze(1).cpu().numpy())
        all_wt.extend(w_t.squeeze(1).cpu().numpy())

all_preds = np.array(all_preds)
all_labs  = np.array(all_labs)
all_probs = np.array(all_probs)
all_wc    = np.array(all_wc)
all_wt    = np.array(all_wt)

print(f"\n  ── MAPA-CLIP-FDB (Optimal Transport + HSIC Alignment) ──")
print(f"  Overall Accuracy  : {accuracy_score(all_labs, all_preds):.4f}")
print(f"  Balanced Accuracy : {balanced_accuracy_score(all_labs, all_preds):.4f}")
print(f"  Macro F1          : {f1_score(all_labs, all_preds, average='macro'):.4f}")
print(f"  Weighted F1       : {f1_score(all_labs, all_preds, average='weighted'):.4f}")
print(f"\n  ── Zero-Shot CLIP Baseline ──")
print(f"  Overall Accuracy  : {accuracy_score(zs_labs, zs_preds):.4f}")
print(f"  Balanced Accuracy : {balanced_accuracy_score(zs_labs, zs_preds):.4f}")
print(f"  Macro F1          : {f1_score(zs_labs, zs_preds, average='macro'):.4f}")
print(f"\n  ── Per-Class Report ──")
print(classification_report(all_labs, all_preds, target_names=classes, digits=4))

alpha_val = torch.sigmoid(model.alpha).item()
print(f"  Learned α = {alpha_val:.4f}  ({alpha_val:.4f}·IT + {1-alpha_val:.4f}·MT)")
print(f"\n  Fuzzy branch weights (mean ± std):")
print(f"    CNN (DenseNet) : {all_wc.mean():.4f} ± {all_wc.std():.4f}")
print(f"    ViT (CLIP)     : {all_wt.mean():.4f} ± {all_wt.std():.4f}")

# =============================================================================
# 17. ABLATION
# =============================================================================
print(f"\n{'='*60}\nABLATION STUDY\n{'='*60}")

model.eval()
it_preds, im_preds, vit_only_preds, cnn_only_preds = [], [], [], []

with torch.no_grad():
    for clip_imgs, dense_imgs, meta, labs in test_loader:
        clip_imgs  = clip_imgs.to(DEVICE)
        dense_imgs = dense_imgs.to(DEVICE)
        meta_d     = meta.to(DEVICE)
        _, L_it, L_im = model(clip_imgs, dense_imgs, meta_d, mode="eval")
        it_preds.extend(L_it.argmax(1).cpu().numpy())
        im_preds.extend(L_im.argmax(1).cpu().numpy())
        T   = model._encode_text()
        tau = model.log_scale.exp()
        F_t_n = F.normalize(model._encode_vit(clip_imgs), dim=-1)
        vit_only_preds.extend((tau * (F_t_n @ T.T)).argmax(1).cpu().numpy())
        F_c_n = F.normalize(model.densenet_branch(dense_imgs), dim=-1)
        cnn_only_preds.extend((tau * (F_c_n @ T.T)).argmax(1).cpu().numpy())

it_preds       = np.array(it_preds)
im_preds       = np.array(im_preds)
vit_only_preds = np.array(vit_only_preds)
cnn_only_preds = np.array(cnn_only_preds)

print(f"  {'Branch':<42} {'Acc':>6} {'BalAcc':>8} {'MacroF1':>9}")
print(f"  {'-'*67}")
for name, preds in [
    ("Zero-Shot CLIP (ViT-B/16)",              zs_preds),
    ("ViT-only (CoOp + CLIP)",                 vit_only_preds),
    ("CNN-only (DenseNet)",                    cnn_only_preds),
    ("Fuzzy Dual-Branch img-text",             it_preds),
    ("Metadata-Text only",                     im_preds),
    ("MAPA-CLIP-FDB (OT+HSIC alignment)",      all_preds),
]:
    print(f"  {name:<42} "
          f"{accuracy_score(all_labs, preds):>6.4f} "
          f"{balanced_accuracy_score(all_labs, preds):>8.4f} "
          f"{f1_score(all_labs, preds, average='macro'):>9.4f}")

# =============================================================================
# 18. VISUALISATIONS
# =============================================================================
print("\nGenerating plots …")

fig, axes = plt.subplots(2, 4, figsize=(26, 12))
fig.suptitle(
    f"MAPA-CLIP-FDB (OT+HSIC Alignment)  —  PAD-UFES-20  —  {N_SHOTS}-shot"
    f"  |  RUN_SEED={RUN_SEED}",
    fontsize=12, fontweight="bold",
)
ep_range = range(1, EPOCHS + 1)

ax = axes[0, 0]
for key, label in [("loss","Total"),("cls","Classification"),
                    ("ot","OT"),("hsic","HSIC"),("moment","Moment"),
                    ("recon","Recon"),("trip","Triplet")]:
    ax.plot(ep_range, hist[key], lw=1.5, label=label)
ax.set_title("Training Loss Components"); ax.set_xlabel("Epoch")
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(ep_range, hist["acc"], color="steelblue", lw=2.0)
ax.axhline(best_acc, color="red", ls="--", lw=1.5, label=f"Best={best_acc:.4f} @ep{best_ep}")
ax.set_title("Test Accuracy vs. Epoch"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 2]
ax.plot(ep_range, hist["w_cnn_mean"], color="darkorange", lw=1.8, label="w_CNN")
ax.plot(ep_range, hist["w_vit_mean"], color="steelblue",  lw=1.8, label="w_ViT")
ax.axhline(0.5, color="gray", ls=":", lw=1.0)
ax.set_title("Fuzzy Branch Weights"); ax.set_xlabel("Epoch")
ax.set_ylabel("Mean Weight"); ax.set_ylim(0, 1)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 3]
ax.hist(all_wc, bins=40, color="darkorange", alpha=0.6, label="w_CNN")
ax.hist(all_wt, bins=40, color="steelblue",  alpha=0.6, label="w_ViT")
ax.set_title("Weight Distributions")
ax.set_xlabel("Weight"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

sns.heatmap(confusion_matrix(all_labs, all_preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=axes[1, 0])
axes[1, 0].set_title("MAPA-CLIP-FDB (OT+HSIC)")

sns.heatmap(confusion_matrix(zs_labs, zs_preds), annot=True, fmt="d", cmap="Oranges",
            xticklabels=classes, yticklabels=classes, ax=axes[1, 1])
axes[1, 1].set_title("Zero-Shot CLIP")

f1_zs_pc  = f1_score(zs_labs,  zs_preds,       average=None, labels=list(range(N_CLASSES)))
f1_vit_pc = f1_score(all_labs, vit_only_preds, average=None, labels=list(range(N_CLASSES)))
f1_cnn_pc = f1_score(all_labs, cnn_only_preds, average=None, labels=list(range(N_CLASSES)))
f1_it_pc  = f1_score(all_labs, it_preds,       average=None, labels=list(range(N_CLASSES)))
f1_im_pc  = f1_score(all_labs, im_preds,       average=None, labels=list(range(N_CLASSES)))
f1_fdb_pc = f1_score(all_labs, all_preds,      average=None, labels=list(range(N_CLASSES)))

x = np.arange(N_CLASSES); w = 0.13
for k, (vals, label, color) in enumerate([
    (f1_zs_pc,  "Zero-Shot",  "silver"),
    (f1_vit_pc, "ViT-only",   "slategray"),
    (f1_cnn_pc, "CNN-only",   "peru"),
    (f1_it_pc,  "FDB Img-Txt","steelblue"),
    (f1_im_pc,  "Meta-Text",  "seagreen"),
    (f1_fdb_pc, "MAPA-OT+HSIC","crimson"),
]):
    axes[1, 2].bar(x + (k - 2.5) * w, vals, w, label=label, color=color, alpha=0.85)
axes[1, 2].set_xticks(x); axes[1, 2].set_xticklabels(classes)
axes[1, 2].set_title("Per-Class F1 Comparison")
axes[1, 2].set_ylim(0, 1.05)
axes[1, 2].legend(fontsize=7); axes[1, 2].grid(axis="y", alpha=0.3)

summary = [
    ["Zero-Shot CLIP",
     f"{accuracy_score(zs_labs, zs_preds):.4f}",
     f"{balanced_accuracy_score(zs_labs, zs_preds):.4f}",
     f"{f1_score(zs_labs, zs_preds, average='macro'):.4f}"],
    ["ViT-only (CoOp)",
     f"{accuracy_score(all_labs, vit_only_preds):.4f}",
     f"{balanced_accuracy_score(all_labs, vit_only_preds):.4f}",
     f"{f1_score(all_labs, vit_only_preds, average='macro'):.4f}"],
    ["CNN-only",
     f"{accuracy_score(all_labs, cnn_only_preds):.4f}",
     f"{balanced_accuracy_score(all_labs, cnn_only_preds):.4f}",
     f"{f1_score(all_labs, cnn_only_preds, average='macro'):.4f}"],
    ["FDB Img-Text",
     f"{accuracy_score(all_labs, it_preds):.4f}",
     f"{balanced_accuracy_score(all_labs, it_preds):.4f}",
     f"{f1_score(all_labs, it_preds, average='macro'):.4f}"],
    ["Meta-Text",
     f"{accuracy_score(all_labs, im_preds):.4f}",
     f"{balanced_accuracy_score(all_labs, im_preds):.4f}",
     f"{f1_score(all_labs, im_preds, average='macro'):.4f}"],
    ["MAPA (OT+HSIC)",
     f"{accuracy_score(all_labs, all_preds):.4f}",
     f"{balanced_accuracy_score(all_labs, all_preds):.4f}",
     f"{f1_score(all_labs, all_preds, average='macro'):.4f}"],
]
axes[1, 3].axis("off")
tbl = axes[1, 3].table(
    cellText=summary, colLabels=["Method", "Accuracy", "Bal.Acc", "MacroF1"],
    cellLoc="center", loc="center",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.0, 2.2)
for col in range(4):
    tbl[(6, col)].set_facecolor("#d4edda")
axes[1, 3].set_title("Performance Summary", pad=20)

plt.tight_layout()
fig_path = f"{OUTPUT_DIR}/mapa_clip_fdb_run{RUN_SEED}_results.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Plots saved → {fig_path}")

# =============================================================================
# 19. EMBEDDING PCA
# =============================================================================
try:
    from sklearn.decomposition import PCA
    model.eval()
    Z_list, I_list, Fc_list, Ft_list, labs_vis = [], [], [], [], []
    with torch.no_grad():
        for clip_imgs, dense_imgs, meta, labs in test_loader:
            clip_imgs  = clip_imgs.to(DEVICE)
            dense_imgs = dense_imgs.to(DEVICE)
            I, F_c, F_t, _, _ = model._encode_images(clip_imgs, dense_imgs)
            M, z = model._encode_meta(meta.to(DEVICE), recon=False)
            Z_list.append(z.cpu().numpy()); I_list.append(I.cpu().numpy())
            Fc_list.append(F.normalize(F_c, dim=-1).cpu().numpy())
            Ft_list.append(F.normalize(F_t, dim=-1).cpu().numpy())
            labs_vis.extend(labs.numpy())

    labs_vis = np.array(labs_vis)
    pca_z  = PCA(2, random_state=GLOBAL_SEED).fit_transform(np.concatenate(Z_list))
    pca_I  = PCA(2, random_state=GLOBAL_SEED).fit_transform(np.concatenate(I_list))
    pca_fc = PCA(2, random_state=GLOBAL_SEED).fit_transform(np.concatenate(Fc_list))
    pca_ft = PCA(2, random_state=GLOBAL_SEED).fit_transform(np.concatenate(Ft_list))

    colors = plt.cm.tab10(np.linspace(0, 1, N_CLASSES))
    fig2, axs = plt.subplots(1, 4, figsize=(22, 5))
    fig2.suptitle(f"PCA of Embeddings (OT+HSIC Alignment)", fontsize=13)
    for ax, pca_data, title in zip(
        axs, [pca_z, pca_fc, pca_ft, pca_I],
        ["Meta Latent z", "CNN F_c", "ViT F_t", "Fused I"],
    ):
        for k, cls in enumerate(classes):
            mask = labs_vis == k
            ax.scatter(pca_data[mask, 0], pca_data[mask, 1],
                       c=[colors[k]], label=cls, alpha=0.6, s=18)
        ax.set_title(title, fontsize=9); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    emb_path = f"{OUTPUT_DIR}/embeddings_pca_run{RUN_SEED}.png"
    plt.savefig(emb_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Embedding PCA saved → {emb_path}")
except Exception as e:
    print(f"  [Warning] PCA skipped: {e}")

# =============================================================================
# 20. SAVE PREDICTIONS
# =============================================================================
pred_df = pd.DataFrame({
    "img_id":      df.iloc[TEST_IDX][img_col].values,
    "true_label":  [i2c[i] for i in all_labs],
    "pred_label":  [i2c[i] for i in all_preds],
    "correct":     (all_preds == all_labs),
    "fuzzy_w_cnn": all_wc,
    "fuzzy_w_vit": all_wt,
    "run_seed":    RUN_SEED,
    **{f"prob_{c}": all_probs[:, k] for k, c in enumerate(classes)},
})
pred_path = f"{OUTPUT_DIR}/test_predictions_run{RUN_SEED}.csv"
pred_df.to_csv(pred_path, index=False)
print(f"  Predictions saved → {pred_path}")

# =============================================================================
# 21. DONE
# =============================================================================
print(f"\n{'='*60}\nDONE\n{'='*60}")
print(f"  GLOBAL_SEED : {GLOBAL_SEED}")
print(f"  RUN_SEED    : {RUN_SEED}")
print(f"  Best accuracy        : {best_acc:.4f}  (epoch {best_ep})")
print(f"  Zero-shot baseline   : {accuracy_score(zs_labs, zs_preds):.4f}")
print(f"  Improvement          : +{best_acc - accuracy_score(zs_labs, zs_preds):.4f}")
print(f"\n  ★ Novel Alignment: Optimal Transport + HSIC (NO contrastive loss)")
print(f"    This approach aligns vision-text distributions geometrically")
print(f"    while maximizing within-class dependence via HSIC.")